In [ ]:
# 01. Imports & Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")
os.makedirs('outputs', exist_ok=True)
def savefig_and_show(filename):
    plt.savefig(f'outputs/{filename}.png', bbox_inches='tight', dpi=120)
    plt.show()
    plt.clf()

sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 13
sns.set_palette(sns.color_palette("tab10", 10))

# 02. Data Loading & Initial Inspection
df = pd.read_csv('hr_attrition_cleaned_data.csv')
df.columns = df.columns.str.strip().str.lower()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Sample data:\n", df.head())

# 03. Missing Value Analysis & Heatmap
print("Missing values per column:\n", df.isnull().sum())
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Value Heatmap',fontweight='bold', color='black')
plt.tight_layout()
savefig_and_show('missing_value_heatmap')

# 04. Remove Irrelevant Columns (skip if not in your data)
cols_to_drop = [c for c in ["employeecount", "employeenumber", "over18", "standardhours"] if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

# 05. Data Description Summary
print("\n--- Data Description Summary ---")
print(df.describe(include='all').T)

# 06. Impute Missing Values
for col in df.select_dtypes('number').columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes('object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# 07. Target Variable: Attrition Visuals & Percentage
attr_counts = df['attrition'].value_counts()
attr_perc = df['attrition'].value_counts(normalize=True) * 100
print("\nAttrition Percentages (%):")
print(attr_perc.round(2))
plt.figure(figsize=(6,4))
sns.barplot(x=attr_counts.index, y=attr_perc.values, palette="tab10")
plt.title("Attrition Percentage",fontweight='bold', color='black')
plt.xlabel("Attrition")
plt.ylabel("Percentage (%)")
for i, v in enumerate(attr_perc.values):
    plt.text(i, v+1, f"{v:.1f}%", ha='center', color='black', fontweight='bold')
plt.tight_layout()
savefig_and_show('attrition_percentage_bar')


plt.figure(figsize=(6,6))
plt.pie(attr_perc, labels=attr_perc.index,
        autopct='%1.1f%%', colors=['#43a2ca', '#fd8d3c'], startangle=140)
plt.title("Attrition Breakdown",fontweight='bold', color='black')
plt.tight_layout()
savefig_and_show('attrition_percentage_pie')

# 08. Outlier Removal for Numerics
df_clean = df.copy()
for col in df.select_dtypes('number').columns:
    if col == 'attrition': continue
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
print("Shape after outlier removal:", df_clean.shape)

# 09. Univariate Numeric EDA (Histograms)
for col in df_clean.select_dtypes('number').columns:
    bins = 30
    counts, bin_edges = np.histogram(df_clean[col], bins=bins)
    max_idx = np.argmax(counts)
    peak_range = (bin_edges[max_idx], bin_edges[max_idx+1])
    peak_count = counts[max_idx]
    plt.figure(figsize=(7,5))
    sns.histplot(df_clean[col], bins=bin_edges, kde=True)
    plt.title(f"{col} Distribution (Peak: {peak_range[0]:.1f}-{peak_range[1]:.1f}, n={peak_count})",fontweight='bold', color='black')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.axvspan(peak_range[0], peak_range[1], color='darkred', alpha=0.25)
    plt.annotate(
        f"Peak {peak_range[0]:.1f}-{peak_range[1]:.1f}",
        xy=((peak_range[0] + peak_range[1])/2, peak_count),
        xytext=((peak_range[0] + peak_range[1])/2, peak_count + 18),
        ha='center', color='darkred', fontsize=13, fontweight='bold',
        arrowprops=dict(facecolor='darkred', arrowstyle='->', linewidth=2)
    )
    plt.tight_layout()
    savefig_and_show(f'distrib_{col}_peakrange')

# 10. Univariate Categorical EDA (Countplots)
palette = sns.color_palette("tab10", 10)
for col in df_clean.select_dtypes('object').columns:
    plt.figure(figsize=(7,5))
    order = df_clean[col].value_counts().index
    ax = sns.countplot(x=col, data=df_clean, palette=palette, order=order)
    plt.title(f"{col} Distribution",fontweight='bold', color='black')
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=30)
    plt.tight_layout()
    for p in ax.patches:
        ax.annotate(
            f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
            ha='center', va='bottom', fontsize=11, color='black', fontweight='bold'
        )
    savefig_and_show(f'distrib_{col}_annotated')

# 11. Numeric Means by Attrition (Pivot Table)
if 'monthlyincome' in df_clean.columns:
    print("[Business Insight] Monthly Income avg by Attrition:")
    print(df_clean.pivot_table(index='attrition', values='monthlyincome', aggfunc='mean'))
if 'totalworkingyears' in df_clean.columns:
    print("[Business Insight] Total Working Years avg by Attrition:")
    print(df_clean.pivot_table(index='attrition', values='totalworkingyears', aggfunc='mean'))

# 12. Boxplots (Numeric by Attrition)
for col in df_clean.select_dtypes('number').columns:
    plt.figure(figsize=(7,5))
    ax = sns.boxplot(x='attrition', y=col, data=df_clean, palette="tab10")
    plt.title(f"{col} by Attrition",fontweight='bold', color='black')
    plt.xlabel("Attrition")
    plt.ylabel(col)
    medians = df_clean.groupby('attrition')[col].median()
    for i, median in enumerate(medians):
        ax.annotate(f"Median: {median:.1f}", xy=(i, median), 
                    xytext=(i, median + (ax.get_ylim()[1] - ax.get_ylim()[0])*0.04),
                    ha='center', color='darkred', fontsize=12, fontweight='bold')
    plt.tight_layout()
    savefig_and_show(f'box_{col}_attrition')

# 13. Stacked Cat Proportion Bars by Attrition
# Robust stacked bar categorical proportion by Attrition (skip problematic columns)
for col in df_clean.select_dtypes('object').columns:
    # Skip if only one unique value or all nans
    valcounts = df_clean[col].nunique(dropna=True)
    if valcounts <= 1:
        print(f"Skipping column {col}: only one unique value.")
        continue
    # Avoid NaN only columns
    if df_clean[col].notna().sum() == 0:
        print(f"Skipping column {col}: all values are NaN.")
        continue
    try:
        prop = (
            df_clean.groupby('attrition')[col]
            .value_counts(normalize=True)
            .rename("proportion")
            .reset_index()
        )
        pivot = prop.pivot(index=col, columns='attrition', values='proportion').fillna(0)
        pivot.plot(kind='bar', stacked=True, figsize=(10,6), colormap='tab10')
        plt.title(f"{col} Proportion by Attrition",fontweight='bold', color='black')
        plt.ylabel('Proportion')
        plt.xlabel(col)
        plt.xticks(rotation=30)
        plt.tight_layout()
        savefig_and_show(f'{col}_proportion_stackedbar')
    except Exception as e:
        print(f"Skipping column {col} due to error: {e}")


# 14. Correlation Matrix
plt.figure(figsize=(15,12))
sns.heatmap(df_clean.select_dtypes('number').corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix",fontweight='bold', color='black')
plt.tight_layout()
savefig_and_show('correlation_matrix')

# 15. Export Cleaned Data
df_clean.to_csv('analyzed_attrition_data.csv', index=False)


## Business Insights

### 1. Attrition Level
- Overall attrition rate is **16.12%**, showing a moderately stable workforce.

### 2. Age & Experience Impact
- Employees who left are **younger** (30 vs 35 median age).
- They also have **lower total working experience** (6 vs 9 years).
- They spend **less time at the company** (3 vs 5 years).

### 3. Salary, Job Level & Financial Factors
- Employees who left earn **significantly less** (₹2857 vs ₹4487 monthly income).
- Highest exits are from **Job Level 1**.
- Attrition employees typically have **stock option level 0** (vs 1 for retained).

### 4. Work Environment & Satisfaction
- Lower environment satisfaction among attrition group (2 vs 3).
- No major difference in job satisfaction, involvement, or performance rating.

### 5. Work-life & Overtime Patterns
- Higher attrition among employees with **No Overtime**, hinting disengagement or low workload.
- Work-life balance median is similar for both groups.

### 6. Distance & Travel
- Employees who left live **farther** from office (9 km vs 7 km).
- Highest attrition in **Travel_Rarely** category (0.65 proportion).

### 7. Department & Job Role Risk
- Highest attrition in **Research & Development**.
- By job role, **Laboratory Technicians** show the highest exit rate.

### 8. Demographic Risk Segments
- Higher attrition among **males**.
- **Single** employees leave more than married employees.

### 9. Most Powerful Predictors of Attrition
Top 5 features correlated with attrition:
1. **Years at company** (0.225)
2. **Total working years** (0.218)
3. **Stock option level** (0.208)
4. **Years with current manager** (0.205)
5. **Years in current role** (0.197)
